In [2]:
import numpy as np
import pandas as pd

np.random.seed(6)

users_num = 10000
mean_user_ctr = 0.2
beta = 20
alpha = mean_user_ctr * beta / (1 - mean_user_ctr)

df = pd.DataFrame()
df['user'] = range(users_num)
df['group'] = np.random.rand(users_num) < 0.5

df['base_user_ctr'] = np.random.beta(alpha, beta, size=users_num)
df['views'] = np.random.lognormal(mean=1, sigma=1, size=users_num).astype(int) + 1
df['clicks'] = np.random.binomial(df['views'], df['base_user_ctr'])

In [11]:
import hashlib

df['bucket'] = df['user'].apply(
    lambda x: int(hashlib.md5((str(x) + 'my_salt').encode()).hexdigest(), 16) % 100
)

In [13]:
# new_df = df.groupby(['group', 'bucket']).sum({'views', 'ctr'}).reset_index()
# new_df['ctr'] = new_df.clicks / new_df.views

# new_df.head()
df['user_ctr'] = df['clicks'] / df['views']

new_df = df.groupby(['group', 'bucket']).agg(
    total_clicks=('clicks', 'sum'),
    total_views=('views', 'sum'),
    mean_user_ctr=('user_ctr', 'mean')  # Способ 2: средний CTR пользователей в бакете
).reset_index()

# Способ 1: групповой CTR (отношение суммы кликов к сумме показов в бакете)
new_df['group_ctr'] = new_df['total_clicks'] / new_df['total_views']

# 5. Считаем среднеквадратичное отклонение (std) для одной из групп (например, group == True)
group_true = new_df[new_df['group'] == True]

std_group_ctr = group_true['group_ctr'].std()
std_mean_user_ctr = group_true['mean_user_ctr'].std()

print(f"STD группового CTR (sum_clicks / sum_views): {std_group_ctr:.5f}")
print(f"STD среднего пользовательского CTR:          {std_mean_user_ctr:.5f}")


STD группового CTR (sum_clicks / sum_views): 0.02771
STD среднего пользовательского CTR:          0.03686
